# 20 - The multi-league pipeline: does the pooled model ship?

Notebook 18 found that one logistic regression pooled over the top 5 leagues
beats a model per league. That was on research data (Understat xG, names
matched on the fly). This notebook re-runs the comparison on the
**pipeline's own data and code**, the gate for adding La Liga, the Bundesliga,
Serie A and Ligue 1 to `ENABLED` (docs/multi-league-plan.md, Phase 1).

- **Data** - `load_data.build_dataset` over E0, SP1, D1, I1, F1 from the
  cached raw files (`refresh_current=False`, so no network): football-data
  results, FotMob xG with football-data / Understat as backup, names through
  `team_names.csv`.
- **Features** - `EloDiff`, `XGDDiffLast5`, `GoalsPerGameDiff`, built per
  competition (`scope="competition"`, P1-04) by the pipeline's builders.
- **Models** - the deployed calibrated LR (`model.make_model`):
  - **EPL-only** - today's model, trained on the Premier League alone;
  - **per league** - the same, one model per league;
  - **pooled** - one model on all five, plus `League_<Div>` indicators
    (E0 baseline), as `train_model` now does.
- **Evaluation** - `evaluate.py`: walk-forward by season, test 22-23 to 25-26,
  per-match log loss, paired bootstrap (5,000 resamples, seed 42).

**Gates:**

1. EPL, 1,520 test matches: pooled vs EPL-only must be **non-inferior**
   (95% CI upper bound < +0.002). If this fails, the pooled model doesn't ship.
2. All top-5 test matches (~7,080): pooled vs per-league must be **better**
   (CI upper bound < 0), replicating nb 18's -0.0028 [-0.0044, -0.0012].

In [1]:
import os
import sys
import tempfile
import warnings
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)  # the pipeline reads data/ relative to the repo root
sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, log_loss

import competitions
from load_data import build_dataset
from features import build_goals_per_game_feature, build_xgd_last5_feature
from elo import build_elo_feature
from model import FEATURES, LEAGUE_COLUMNS, league_columns, make_model
from backtest import bet_profit
from evaluate import (
    walk_forward, per_match_log_loss, paired_bootstrap, seasons_better,
    better, non_inferior, NON_INFERIOR_MARGIN,
)

warnings.filterwarnings("ignore")
pd.set_option("display.precision", 4)
pd.set_option("display.width", 200)

DIVS = ["E0", "SP1", "D1", "I1", "F1"]
TEST_SEASONS = ["22-23", "23-24", "24-25", "25-26"]
LABELS = ["A", "D", "H"]  # model.classes_ order
NAMES = {div: competitions.get(div)["name"] for div in DIVS}

## Build the dataset with the pipeline

In [2]:
with tempfile.TemporaryDirectory() as tmp:
    matches = build_dataset(DIVS, output_path=Path(tmp) / "all_matches.csv", refresh_current=False)

matches = matches.sort_values(["MatchDateTime", "HomeTeam", "AwayTeam"]).reset_index(drop=True)

# 26-27 is in progress; it stays in for feature history, never as a test season
coverage = (
    matches.assign(HasXG=matches["HomeXG"].notna() & matches["AwayXG"].notna())
    .groupby(["Div", "Season"])
    .agg(Games=("FTR", "size"), XG=("HasXG", "mean"))
    .unstack("Season")
)
coverage

Built 9360 matches (100.0% xG coverage, 1 from football-data/Understat backup)


Saved C:\Users\harry\AppData\Local\Temp\tmp76k5rd9f\all_matches.csv


Games                                  XG                              
Season 21-22 22-23 23-24 24-25 25-26 26-27 21-22 22-23 23-24 24-25 25-26 26-27
Div                                                                           
D1       306   306   306   306   306    72   1.0   1.0   1.0   1.0   1.0   1.0
E0       380   380   380   380   380    90   1.0   1.0   1.0   1.0   1.0   1.0
F1       380   380   306   306   306    81   1.0   1.0   1.0   1.0   1.0   1.0
I1       380   380   380   380   380   100   1.0   1.0   1.0   1.0   1.0   1.0
SP1      380   380   380   380   380   109   1.0   1.0   1.0   1.0   1.0   1.0

In [3]:
matches.groupby(["Div", "XGSource"]).size().unstack(fill_value=0)

XGSource,FotMob,Understat
Div,,
D1,1602,0
E0,1990,0
F1,1759,0
I1,2000,0
SP1,2008,1


## Features

Built once over the pooled frame. P1-04 showed each league's features are
bit-identical to building it alone, so the EPL-only reference uses the same
E0 rows.

In [4]:
build_elo_feature(matches)
build_goals_per_game_feature(matches)
build_xgd_last5_feature(matches)

matches = league_columns(matches)
INDICATORS = [c for c in LEAGUE_COLUMNS.values() if c in matches.columns]

test = matches[matches["Season"].isin(TEST_SEASONS)]
e0_test = test[test["Div"] == "E0"]
print(f"{len(test)} test matches, {len(e0_test)} in the Premier League; indicators: {INDICATORS}")

7082 test matches, 1520 in the Premier League; indicators: ['League_SP1', 'League_D1', 'League_I1', 'League_F1']


## Walk-forward predictions

In [5]:
e0 = matches[matches["Div"] == "E0"]

probas = {
    "EPL-only": walk_forward(e0, make_model, FEATURES, TEST_SEASONS),
    "Per league": walk_forward(matches, make_model, FEATURES, TEST_SEASONS, group_by="Div"),
    "Pooled": walk_forward(matches, make_model, FEATURES + INDICATORS, TEST_SEASONS),
    "Pooled, no indicators": walk_forward(matches, make_model, FEATURES, TEST_SEASONS),
}

# Per league on E0 is the EPL-only model, trained on exactly the same rows
assert np.allclose(probas["Per league"].loc[e0_test.index], probas["EPL-only"].loc[e0_test.index])

In [6]:
def demargin(row, prefix):
    odds = np.array([row[f"{prefix}{o}"] for o in LABELS], dtype=float)
    implied = 1 / odds
    return implied / implied.sum()


def bookmaker(frame):
    """De-margined Pinnacle closing odds, falling back to the market-average close."""
    rows = []
    for _, row in frame.iterrows():
        prefix = "PSC" if row[["PSCA", "PSCD", "PSCH"]].notna().all() else "AvgC"
        rows.append(demargin(row, prefix))
    return pd.DataFrame(rows, index=frame.index, columns=LABELS)


probas["Bookmaker"] = bookmaker(test)

# Each league's home/draw/away rates from the seasons before the test season
base = []
for season in TEST_SEASONS:
    prior = matches[matches["Season"] < season]
    rates = prior.groupby("Div")["FTR"].value_counts(normalize=True).unstack()[LABELS]
    held = test[test["Season"] == season]
    base.append(pd.DataFrame(rates.loc[held["Div"]].to_numpy(), index=held.index, columns=LABELS))
probas["Base rates"] = pd.concat(base).loc[test.index]

## Scores by league

In [7]:
def scores(frame, proba):
    proba = proba.loc[frame.index]
    pick = proba.idxmax(axis=1)
    profit = sum(bet_profit(m, p) for (_, m), p in zip(frame.iterrows(), pick))
    return {
        "matches": len(frame),
        "logLoss": log_loss(frame["FTR"], proba[LABELS], labels=LABELS),
        "accuracy": accuracy_score(frame["FTR"], pick) * 100,
        "roi": profit / len(frame) * 100,
    }


models = ["Base rates", "Per league", "Pooled, no indicators", "Pooled", "Bookmaker"]
rows = []
for div in DIVS + ["All"]:
    frame = test if div == "All" else test[test["Div"] == div]
    for name in models:
        rows.append({"League": NAMES.get(div, "All five"), "Model": name, **scores(frame, probas[name])})

by_league = pd.DataFrame(rows)
by_league.pivot(index="League", columns="Model", values="logLoss")[models].loc[[*NAMES.values(), "All five"]]

Model,Base rates,Per league,"Pooled, no indicators",Pooled,Bookmaker
League,,,,,
Premier League,1.0689,0.9799,0.9784,0.9798,0.9605
La Liga,1.0639,0.9911,0.9872,0.9847,0.9595
Bundesliga,1.0749,0.9991,0.9931,0.9935,0.9697
Serie A,1.0869,0.9948,0.9928,0.9928,0.9670
Ligue 1,1.0706,0.9989,0.9982,0.9982,0.9784
All five,1.0731,0.9923,0.9896,0.9894,0.9666


In [8]:
(
    by_league[by_league["Model"].isin(["Per league", "Pooled"])]
    .pivot(index="League", columns="Model", values=["accuracy", "roi"])
    .loc[[*NAMES.values(), "All five"]]
    .round(1)
)

accuracy               roi       
Model          Per league Pooled Per league Pooled
League                                            
Premier League       54.7   54.5        0.4    0.4
La Liga              52.3   53.6       -1.4    1.6
Bundesliga           51.6   51.6       -6.4   -6.6
Serie A              52.1   52.5       -3.5   -2.6
Ligue 1              51.6   51.8       -4.8   -4.1
All five             52.5   52.9       -2.9   -2.0

## Gate 1 - EPL: pooled vs EPL-only (non-inferior)

Per-match log-loss difference, pooled - EPL-only, on the 1,520 Premier League
test matches. Negative = pooled better.

In [9]:
def compare(frame, candidate, reference):
    y = frame["FTR"].to_numpy()
    diff = (
        per_match_log_loss(y, probas[candidate].loc[frame.index][LABELS].to_numpy(), LABELS)
        - per_match_log_loss(y, probas[reference].loc[frame.index][LABELS].to_numpy(), LABELS)
    )
    result = paired_bootstrap(diff)
    return {
        "matches": len(frame),
        "mean": result["mean"],
        "ciLow": result["ci"][0],
        "ciHigh": result["ci"][1],
        "pBetter": result["p_better"],
        "seasonsBetter": seasons_better(
            y, probas[candidate].loc[frame.index][LABELS].to_numpy(),
            probas[reference].loc[frame.index][LABELS].to_numpy(),
            frame["Season"].to_numpy(), labels=LABELS,
        ),
        "better": better(result["ci"]),
        "nonInferior": non_inferior(result["ci"]),
    }


gate1 = compare(e0_test, "Pooled", "EPL-only")
print(f"Pooled - EPL-only on {gate1['matches']} EPL matches: {gate1['mean']:+.4f} "
      f"[{gate1['ciLow']:+.4f}, {gate1['ciHigh']:+.4f}], P(better) {gate1['pBetter']:.2f}, "
      f"better in {gate1['seasonsBetter']}/{len(TEST_SEASONS)} seasons")
print(f"Gate 1 (non-inferior, upper < +{NON_INFERIOR_MARGIN}): {'PASS' if gate1['nonInferior'] else 'FAIL'}")

Pooled - EPL-only on 1520 EPL matches: -0.0001 [-0.0028, +0.0025], P(better) 0.53, better in 2/4 seasons
Gate 1 (non-inferior, upper < +0.002): FAIL


## Gate 2 - top 5: pooled vs per league (better)

In [10]:
gate2 = compare(test, "Pooled", "Per league")
print(f"Pooled - per league on {gate2['matches']} matches: {gate2['mean']:+.4f} "
      f"[{gate2['ciLow']:+.4f}, {gate2['ciHigh']:+.4f}], P(better) {gate2['pBetter']:.2f}, "
      f"better in {gate2['seasonsBetter']}/{len(TEST_SEASONS)} seasons")
print(f"Gate 2 (better, upper < 0): {'PASS' if gate2['better'] else 'FAIL'}")
print("nb 18 (Understat data): -0.0028 [-0.0044, -0.0012]")

Pooled - per league on 7082 matches: -0.0029 [-0.0046, -0.0013], P(better) 1.00, better in 4/4 seasons
Gate 2 (better, upper < 0): PASS
nb 18 (Understat data): -0.0028 [-0.0044, -0.0012]


By league, and the indicators on their own (not gates - for the record):

In [11]:
detail = []
for div in DIVS:
    frame = test[test["Div"] == div]
    detail.append({"Comparison": "Pooled - per league", "League": NAMES[div], **compare(frame, "Pooled", "Per league")})
detail.append({"Comparison": "Pooled - pooled without indicators", "League": "All five",
               **compare(test, "Pooled", "Pooled, no indicators")})
pd.DataFrame(detail).drop(columns=["better", "nonInferior"])

,Comparison,League,matches,mean,ciLow,ciHigh,pBetter,seasonsBetter
0,Pooled - per league,Premier League,1520,-9.1232e-05,-0.0028,0.0025,0.5312,2
1,Pooled - per league,La Liga,1520,-6.4013e-03,-0.0108,-0.0020,0.9980,3
2,Pooled - per league,Bundesliga,1224,-5.6265e-03,-0.0105,-0.0008,0.9896,4
3,Pooled - per league,Serie A,1520,-2.0048e-03,-0.0049,0.0009,0.9090,3
4,Pooled - per league,Ligue 1,1298,-6.9423e-04,-0.0034,0.0022,0.6794,2
5,Pooled - pooled without indicators,All five,7082,-1.9690e-04,-0.0014,0.0010,0.6304,3


## Season by season

In [12]:
pd.DataFrame({
    name: {
        season: log_loss(test.loc[test["Season"] == season, "FTR"],
                         probas[name].loc[test.index[test["Season"] == season]][LABELS], labels=LABELS)
        for season in TEST_SEASONS
    }
    for name in ["Per league", "Pooled", "Bookmaker"]
}).assign(**{"Pooled - per league": lambda t: t["Pooled"] - t["Per league"]})

,Per league,Pooled,Bookmaker,Pooled - per league
22-23,0.9972,0.9935,0.9749,-0.0037
23-24,0.9804,0.9762,0.9525,-0.0042
24-25,0.9871,0.9852,0.9606,-0.0019
25-26,1.0042,1.0024,0.9780,-0.0018


## League indicator coefficients

The pooled model trained on 21-22 to 25-26, coefficients averaged over the
5 calibration folds (log-odds, per outcome, relative to the Premier League).

In [13]:
train = matches[matches["Season"] < "26-27"]
fitted = make_model().fit(train[FEATURES + INDICATORS], train["FTR"])
coef = np.mean([c.estimator.coef_ for c in fitted.calibrated_classifiers_], axis=0)
pd.DataFrame(coef, index=fitted.classes_, columns=FEATURES + INDICATORS).T.round(3)

,A,D,H
EloDiff,-0.003,0.000,0.002
XGDDiffLast5,-0.099,-0.070,0.170
GoalsPerGameDiff,-0.103,0.046,0.057
League_SP1,-0.129,0.076,0.053
League_D1,-0.028,0.028,0.000
League_I1,0.013,0.108,-0.121
League_F1,0.018,0.008,-0.026


## After the gate (not gates - chosen after seeing gate 1)

Gate 1 failed on its upper bound, with a mean of about zero. Two questions for
the owner's decision, answered here and **not** used to pass anything:

1. Could a model exactly as good as today's pass this gate on 1,520 matches?
2. What would the alternatives look like?

In [14]:
y = e0_test["FTR"].to_numpy()
diff = (
    per_match_log_loss(y, probas["Pooled"].loc[e0_test.index][LABELS].to_numpy(), LABELS)
    - per_match_log_loss(y, probas["EPL-only"].loc[e0_test.index][LABELS].to_numpy(), LABELS)
)
sd = diff.std(ddof=1)
se = sd / np.sqrt(len(diff))
print(f"Per-match difference SD {sd:.4f}; SE of the mean {se:.5f}; CI half-width ~{1.96 * se:.4f}")
print(f"To pass at n = {len(diff)}, the pooled model must be better by > {1.96 * se - NON_INFERIOR_MARGIN:.4f}")
print(f"A model exactly as good as today's needs n ~ {(1.96 * sd / NON_INFERIOR_MARGIN) ** 2:,.0f} EPL matches to pass")

Per-match difference SD 0.0515; SE of the mean 0.00132; CI half-width ~0.0026
To pass at n = 1520, the pooled model must be better by > 0.0006
A model exactly as good as today's needs n ~ 2,549 EPL matches to pass


In [15]:
# Hybrid: today's EPL-only model for the Premier League, the pooled model for the other four
probas["Hybrid"] = probas["Pooled"].copy()
probas["Hybrid"].loc[e0_test.index] = probas["EPL-only"].loc[e0_test.index]

after = pd.DataFrame([
    {"Candidate": "Pooled, no indicators", "Reference": "EPL-only", "Matches": "EPL",
     **compare(e0_test, "Pooled, no indicators", "EPL-only")},
    {"Candidate": "Pooled, no indicators", "Reference": "Per league", "Matches": "All five",
     **compare(test, "Pooled, no indicators", "Per league")},
    {"Candidate": "Hybrid", "Reference": "Per league", "Matches": "All five",
     **compare(test, "Hybrid", "Per league")},
])
after

,Candidate,Reference,Matches,matches,mean,ciLow,ciHigh,pBetter,seasonsBetter,better,nonInferior
0,"Pooled, no indicators",EPL-only,EPL,1520,-0.0014,-0.0050,0.0020,0.7882,2,False,True
1,"Pooled, no indicators",Per league,All five,7082,-0.0027,-0.0049,-0.0006,0.9936,4,True,True
2,Hybrid,Per league,All five,7082,-0.0029,-0.0044,-0.0014,0.9998,4,True,True



## Conclusion

**Gate 1 fails, so the pooled model doesn't ship as planned.** On the 1,520
Premier League test matches, pooled vs today's EPL-only model is
**-0.0001 [-0.0028, +0.0025]**: P(better) 0.53, better in 2 of 4 seasons. The
pooled model is no worse on average, but the upper bound misses the +0.002
non-inferiority margin.

**Gate 2 passes, replicating notebook 18 on the pipeline's own data.**
Pooled vs per league over all 7,082 top-5 test matches is
**-0.0029 [-0.0046, -0.0013]**, better in all 4 seasons (nb 18:
-0.0028 [-0.0044, -0.0012]). The gain is in the other leagues, and the
Premier League is unchanged:

| League | Per league | Pooled | Pooled - per league [95% CI] | Bookmaker |
|---|---|---|---|---|
| Premier League | 0.9799 | 0.9798 | -0.0001 [-0.0028, +0.0025] | 0.9605 |
| La Liga | 0.9911 | 0.9847 | -0.0064 [-0.0108, -0.0020] | 0.9595 |
| Bundesliga | 0.9991 | 0.9935 | -0.0056 [-0.0105, -0.0008] | 0.9697 |
| Serie A | 0.9948 | 0.9928 | -0.0020 [-0.0049, +0.0009] | 0.9670 |
| Ligue 1 | 0.9989 | 0.9982 | -0.0007 [-0.0034, +0.0022] | 0.9784 |
| **All five** | **0.9923** | **0.9894** | **-0.0029 [-0.0046, -0.0013]** | 0.9666 |

**Why gate 1 failed: it can't be passed by a model that is merely as good.**
The per-match difference has SD 0.0515, so on 1,520 matches the CI
half-width is about ±0.0026, wider than the 0.002 margin. To pass, the pooled
model would have to be **better** on the EPL by more than 0.0006, or the test
would need about 2,550 EPL matches. The plan sized the margin from the
pooled-vs-per-league interval over all 7,082 matches, not the EPL's 1,520.

**Options (the owner's decision; nothing here changes `ENABLED`):**

1. **Hybrid: today's EPL-only model for the Premier League, pooled for the
   other four.** EPL predictions are exactly today's, so gate 1 holds by
   construction. Over all five leagues it keeps the whole gain:
   -0.0029 [-0.0044, -0.0014], better in 4/4 seasons. The cost is two models
   to train each run instead of one.
2. **Accept the pooled model on the evidence.** The mean EPL change is -0.0001,
   and P1-04 showed the EPL's features are bit-identical either way. This
   means overruling the gate (or widening the margin for gates on a single
   league's ~1,500 matches), which only the owner can do.
3. **Pooled without the league indicators.** It passes gate 1 by a hair
   (+0.0020 upper bound, -0.0014 mean) and still passes gate 2
   (-0.0027 [-0.0049, -0.0006]). But it was picked **after** gate 1 failed,
   the indicators-vs-none difference is noise (-0.0002 [-0.0014, +0.0010]),
   and a pass this close is a coin flip on resampling. Not recommended.

**Other findings:**
- **Flat-stake ROI** at average opening odds rises from -2.9% to -2.0% pooled.
  La Liga goes from -1.4% to +1.6%; the Premier League stays at +0.4%.
- **Indicator coefficients** are small and plausible: Serie A has more draws
  and fewer home wins; La Liga has fewer away wins.
- **xG coverage** is 100% in every league-season, all FotMob except one
  Understat match (SP1 23-24, Mallorca v Girona).


**Decision (owner, 2026-11-03): option 1, the hybrid.** Each competition's
`model` in `competitions.py` is `"own"` for E0 and `"pooled"` for the rest;
`model.train_models` fits both. The pipeline's own backtest gives exactly
this notebook's hybrid: the E0 section is identical to the EPL-only
backtest, and the other leagues have their pooled figures.


---

# Promoted-team start, per league (P1-08)

Today a promoted club keeps its old rating (`promoted_elo = None`): 1500 if
it's new to the data, or a stale rating from its last stint up. Notebook 19
found starting EPL promoted clubs lower helped only slightly, so the EPL kept
`None`. Other leagues may differ, because the gap between their top two
divisions may be bigger.

- **Candidate:** a promoted club (not in the league the season before) starts
  at `1500 - P`, `P` in {100, 150, 200}.
- **Models:** the shipped setup (`model.walk_forward_models`): E0 on its own
  model, the rest pooled. Only Elo changes, so the other features are reused.
- **Honest tuning:** `P` is chosen on 22-23/23-24 and judged on 24-25/25-26.
- **Pooled `P` first** (one value for every league, the bigger sample), then
  **per-league `P_L`**, changing only that league.
- **Decision, per league, on all of its 24-25/25-26 matches:**
  - adopt `P_L` if it beats both `None` and the pooled value (CI upper < 0);
  - otherwise adopt the pooled value if it beats `None` in that league;
  - otherwise keep `None`.
- **Also reported:** promoted clubs' first 10 games and their whole season.

In [16]:
from elo import LEAGUE_BASE_ELO
from model import walk_forward_models

P_GRID = [100, 150, 200]
TUNE, REPORT = ["22-23", "23-24"], ["24-25", "25-26"]
ALL_TEST = TUNE + REPORT


def promoted_flags(frame):
    """Per match: does it involve a promoted club, and in that club's first 10 games?"""

    seasons = sorted(frame["Season"].unique())
    previous = dict(zip(seasons[1:], seasons[:-1]))
    played = {
        (div, season): set(group["HomeTeam"]) | set(group["AwayTeam"])
        for (div, season), group in frame.groupby(["Div", "Season"])
    }

    def promoted(div, season, team):
        return season in previous and team not in played.get((div, previous[season]), set())

    games = {}  # (div, season, team) -> games so far
    any_promoted, first10 = [], []

    for _, m in frame.iterrows():
        hit, early = False, False
        for team in (m["HomeTeam"], m["AwayTeam"]):
            key = (m["Div"], m["Season"], team)
            games[key] = games.get(key, 0) + 1
            if promoted(*key):
                hit = True
                early = early or games[key] <= 10
        any_promoted.append(hit)
        first10.append(early)

    return pd.Series(any_promoted, index=frame.index), pd.Series(first10, index=frame.index)


matches["Promoted"], matches["PromotedFirst10"] = promoted_flags(matches)


def run(promoted_elo):
    """Walk-forward probabilities with Elo rebuilt for ``promoted_elo``."""

    frame = matches.copy()
    build_elo_feature(frame, promoted_elo=promoted_elo)
    return walk_forward_models(frame, ALL_TEST)


configs = {"None": None}
configs.update({f"P={p} all": LEAGUE_BASE_ELO - p for p in P_GRID})
configs.update({f"P={p} {div}": {div: LEAGUE_BASE_ELO - p} for div in DIVS for p in P_GRID})

runs = {name: run(value) for name, value in configs.items()}

promoted_test = matches.loc[matches["Season"].isin(REPORT)]
print(f"{len(runs)} configurations. Report seasons: {len(promoted_test)} matches, "
      f"{promoted_test['Promoted'].sum()} involving a promoted club, "
      f"{promoted_test['PromotedFirst10'].sum()} in its first 10 games")
promoted_test.groupby("Div")[["Promoted", "PromotedFirst10"]].sum().T

19 configurations. Report seasons: 3504 matches, 972 involving a promoted club, 263 in its first 10 games


Div,D1,E0,F1,I1,SP1
Promoted,132,216,192,216,216
PromotedFirst10,39,55,56,58,55


## Choose `P` on 22-23/23-24

In [17]:
def season_loss(name, frame):
    return log_loss(frame["FTR"], runs[name].loc[frame.index][LABELS], labels=LABELS)


tune_all = matches[matches["Season"].isin(TUNE)]
tune_table = {"All five": {name: season_loss(name, tune_all) for name in ["None"] + [f"P={p} all" for p in P_GRID]}}
for div in DIVS:
    frame = tune_all[tune_all["Div"] == div]
    tune_table[NAMES[div]] = {name: season_loss(name, frame) for name in ["None"] + [f"P={p} {div}" for p in P_GRID]}

pooled_p = min(P_GRID, key=lambda p: tune_table["All five"][f"P={p} all"])
league_p = {div: min(P_GRID, key=lambda p: tune_table[NAMES[div]][f"P={p} {div}"]) for div in DIVS}

print(f"Pooled choice: P = {pooled_p}")
print("Per-league choices:", {div: p for div, p in league_p.items()})
pd.DataFrame({
    league: {("None" if k == "None" else k.split()[0]): v for k, v in row.items()}
    for league, row in tune_table.items()
}).T.round(4)

Pooled choice: P = 150
Per-league choices: {'E0': 150, 'SP1': 150, 'D1': 200, 'I1': 150, 'F1': 150}


,None,P=100,P=150,P=200
All five,0.9846,0.9829,0.9826,0.9829
Premier League,0.9537,0.9521,0.9520,0.9527
La Liga,0.9807,0.9800,0.9798,0.9802
Bundesliga,0.9901,0.9875,0.9867,0.9863
Serie A,0.9966,0.9945,0.9942,0.9943
Ligue 1,1.0051,1.0032,1.0028,1.0029


## Judge on 24-25/25-26

In [18]:
report = matches[matches["Season"].isin(REPORT)]


def diff_ci(candidate, reference, frame):
    y = frame["FTR"].to_numpy()
    diff = (
        per_match_log_loss(y, runs[candidate].loc[frame.index][LABELS].to_numpy(), LABELS)
        - per_match_log_loss(y, runs[reference].loc[frame.index][LABELS].to_numpy(), LABELS)
    )
    result = paired_bootstrap(diff)
    return result["mean"], result["ci"]


def fmt(mean, ci):
    return f"{mean:+.4f} [{ci[0]:+.4f}, {ci[1]:+.4f}]"


rows, decisions = [], {}
pooled_name = f"P={pooled_p} all"

for div in DIVS + ["All"]:
    frame = report if div == "All" else report[report["Div"] == div]
    league_name = None if div == "All" else f"P={league_p[div]} {div}"

    comparisons = [("Pooled P - None", pooled_name, "None")]
    if league_name:
        comparisons += [("P_L - None", league_name, "None"), ("P_L - pooled P", league_name, pooled_name)]

    for label, candidate, reference in comparisons:
        row = {"League": NAMES.get(div, "All five"), "Comparison": label,
               "P": pooled_p if candidate == pooled_name else league_p.get(div)}
        for subset, mask in [("All matches", slice(None)), ("Promoted, season", frame["Promoted"]),
                             ("Promoted, first 10", frame["PromotedFirst10"])]:
            part = frame[mask]
            mean, ci = diff_ci(candidate, reference, part)
            row[subset] = fmt(mean, ci)
            if subset == "All matches":
                row["_ci"] = ci
        row["n"] = len(frame)
        rows.append(row)

    if div != "All":
        by = {r["Comparison"]: r["_ci"] for r in rows if r["League"] == NAMES[div]}
        if better(by["P_L - None"]) and better(by["P_L - pooled P"]):
            decisions[div] = LEAGUE_BASE_ELO - league_p[div]
        elif better(by["Pooled P - None"]):
            decisions[div] = LEAGUE_BASE_ELO - pooled_p
        else:
            decisions[div] = None

results = pd.DataFrame(rows).drop(columns="_ci")
results

,League,Comparison,P,All matches,"Promoted, season","Promoted, first 10",n
0,Premier League,Pooled P - None,150,"+0.0008 [-0.0025, +0.0040]","+0.0039 [-0.0073, +0.0150]","+0.0065 [-0.0266, +0.0400]",760
1,Premier League,P_L - None,150,"+0.0008 [-0.0025, +0.0040]","+0.0039 [-0.0073, +0.0150]","+0.0065 [-0.0266, +0.0400]",760
2,Premier League,P_L - pooled P,150,"+0.0000 [+0.0000, +0.0000]","+0.0000 [+0.0000, +0.0000]","+0.0000 [+0.0000, +0.0000]",760
3,La Liga,Pooled P - None,150,"-0.0018 [-0.0045, +0.0010]","-0.0052 [-0.0146, +0.0042]","-0.0164 [-0.0440, +0.0125]",760
4,La Liga,P_L - None,150,"-0.0017 [-0.0044, +0.0014]","-0.0058 [-0.0157, +0.0042]","-0.0182 [-0.0465, +0.0116]",760
5,La Liga,P_L - pooled P,150,"+0.0002 [-0.0004, +0.0008]","-0.0006 [-0.0017, +0.0007]","-0.0018 [-0.0034, +0.0000]",760
6,Bundesliga,Pooled P - None,150,"-0.0032 [-0.0075, +0.0010]","-0.0120 [-0.0308, +0.0062]","-0.0355 [-0.0835, +0.0152]",612
7,Bundesliga,P_L - None,200,"-0.0035 [-0.0094, +0.0026]","-0.0132 [-0.0398, +0.0126]","-0.0373 [-0.1046, +0.0337]",612
8,Bundesliga,P_L - pooled P,200,"-0.0002 [-0.0022, +0.0018]","-0.0012 [-0.0093, +0.0069]","-0.0018 [-0.0226, +0.0206]",612
9,Serie A,Pooled P - None,150,"-0.0019 [-0.0053, +0.0014]","-0.0082 [-0.0197, +0.0032]","-0.0149 [-0.0452, +0.0164]",760


In [19]:
pd.DataFrame({
    "League": [NAMES[d] for d in DIVS],
    "Pooled P chosen": pooled_p,
    "P_L chosen": [league_p[d] for d in DIVS],
    "promoted_elo": [decisions[d] for d in DIVS],
})

,League,Pooled P chosen,P_L chosen,promoted_elo
0,Premier League,150,150,None
1,La Liga,150,150,None
2,Bundesliga,150,200,None
3,Serie A,150,150,None
4,Ligue 1,150,150,None



## Promoted-team start - conclusion

**No league restarts promoted clubs lower; `promoted_elo` stays `None`
everywhere.** On the tuning seasons a lower start looked better in every
league (P = 150 for four leagues, 200 for the Bundesliga). On the report
seasons no league's gain was clear of zero:

| League | P (chosen on 22-23/23-24) | All matches, 24-25/25-26 | Promoted clubs' season | Their first 10 games | promoted_elo |
|---|---|---|---|---|---|
| Premier League | 150 | +0.0008 [-0.0025, +0.0040] | +0.0039 [-0.0073, +0.0150] | +0.0065 [-0.0266, +0.0400] | None |
| La Liga | 150 | -0.0017 [-0.0044, +0.0014] | -0.0058 [-0.0157, +0.0042] | -0.0182 [-0.0465, +0.0116] | None |
| Bundesliga | 200 | -0.0035 [-0.0094, +0.0026] | -0.0132 [-0.0398, +0.0126] | -0.0373 [-0.1046, +0.0337] | None |
| Serie A | 150 | -0.0020 [-0.0053, +0.0014] | -0.0079 [-0.0195, +0.0038] | -0.0144 [-0.0444, +0.0177] | None |
| Ligue 1 | 150 | -0.0002 [-0.0041, +0.0036] | +0.0013 [-0.0109, +0.0131] | -0.0073 [-0.0399, +0.0239] | None |
| **All five, pooled P = 150** | 150 | -0.0013 [-0.0029, +0.0002] | -0.0036 [-0.0088, +0.0018] | -0.0127 [-0.0280, +0.0021] | |

(Candidate minus `None`, log loss; negative = lower start better. Per-league
rows change only that league. No per-league value beat the pooled one, and
for every league the per-league and pooled P were the same or nearly so.)

- **The Premier League goes the other way** (+0.0008), agreeing with
  notebook 19: `None` stays, as D6 expected.
- **Outside the EPL the direction is consistent but small.** The four other
  leagues all point to a lower start, the gain is concentrated in promoted
  clubs' first 10 games (about 1-4%, as in nb 19), and the pooled test over
  all five just misses (upper bound +0.0002). That is suggestive but doesn't
  meet the rule, so nothing is adopted.
- **Worth re-running** once 26-27 finishes: a fifth test season adds about
  1,750 matches and ~15 promoted clubs, which is roughly what the pooled
  interval needs to settle the question either way. The test is now cheap:
  `promoted_elo` per league lives in `competitions.py` and
  `build_elo_feature(promoted_elo={Div: value})` overrides it.


---

# Portugal and the Netherlands (P2-02)

Adding the Primeira Liga (P1) and the Eredivisie (N1) to the shipped setup:
the Premier League on its own model, every other league on one pooled model
with `League_<Div>` indicators. Data from P2-01: 100% FotMob xG for both,
21-22 to 26-27.

**Gates (docs/multi-league-plan.md, Phase 2), test 22-23 to 25-26:**

1. **Top 5:** the 7-league pooled model must be **non-inferior** to the
   5-league pooled model on the ~7,080 top-5 test matches (CI upper < +0.002).
2. **P1 and N1, each:** pooled must be **better** than the league's own
   model (today's model trained on that league alone), or at least
   non-inferior. A league that fails ships on its own model instead
   (`model="own"` in `competitions.py`); the pooled model covers the rest.
3. **P1 and N1, each:** pooled must be **better** than the league's own
   base rates, with the de-margined bookmaker alongside.
4. **Promoted-team start** for P1 and N1, as P1-08.

**Power, flagged before running:** each league has 4 x 306 = 1,224 test
matches, fewer than the EPL's 1,520 that sank P1-07's gate 1. With a
per-match SD like the EPL's (~0.05) the CI half-width is about +/-0.003, so
gate 2's non-inferiority leg needs pooled to be better by roughly 0.001 or
more; a model that is merely as good fails. The SD is printed below.

In [20]:
NEW = ["P1", "N1"]
DIVS7 = DIVS + NEW
NAMES7 = {div: competitions.get(div)["name"] for div in DIVS7}
KEY = ["Div", "Season", "HomeTeam", "AwayTeam"]

with tempfile.TemporaryDirectory() as tmp:
    matches7 = build_dataset(DIVS7, output_path=Path(tmp) / "all_matches.csv", refresh_current=False)

matches7 = matches7.sort_values(["MatchDateTime", "HomeTeam", "AwayTeam"]).reset_index(drop=True)

build_elo_feature(matches7)
build_goals_per_game_feature(matches7)
build_xgd_last5_feature(matches7)
matches7 = league_columns(matches7)
INDICATORS7 = [c for c in LEAGUE_COLUMNS.values() if c in matches7.columns]

# Features are per competition, so the top 5's are unchanged by adding two leagues
top5_rows = matches7[matches7["Div"].isin(DIVS)].set_index(KEY)[FEATURES].sort_index()
assert top5_rows.equals(matches.set_index(KEY)[FEATURES].sort_index())

test7 = matches7[matches7["Season"].isin(TEST_SEASONS)]
top5_test7 = test7[test7["Div"].isin(DIVS)]
print(f"{len(matches7)} matches; {len(test7)} test matches "
      f"({len(top5_test7)} top 5, {', '.join(f'{d} {(test7.Div == d).sum()}' for d in NEW)}); "
      f"indicators: {INDICATORS7}")
matches7[matches7["Div"].isin(NEW)].groupby(["Div", "Season"]).agg(
    Games=("FTR", "size"), XG=("HomeXG", lambda s: s.notna().mean()),
    XGSource=("XGSource", lambda s: ", ".join(sorted(s.dropna().unique()))),
)

Built 12609 matches (100.0% xG coverage, 1 from football-data/Understat backup)


Saved C:\Users\harry\AppData\Local\Temp\tmp1y677b3a\all_matches.csv


12609 matches; 9530 test matches (7082 top 5, P1 1224, N1 1224); indicators: ['League_SP1', 'League_D1', 'League_I1', 'League_F1', 'League_P1', 'League_N1']


Games   XG XGSource
Div Season                     
N1  21-22     306  1.0   FotMob
    22-23     306  1.0   FotMob
    23-24     306  1.0   FotMob
    24-25     306  1.0   FotMob
    25-26     306  1.0   FotMob
    26-27      99  1.0   FotMob
P1  21-22     306  1.0   FotMob
    22-23     306  1.0   FotMob
    23-24     306  1.0   FotMob
    24-25     306  1.0   FotMob
    25-26     306  1.0   FotMob
    26-27      90  1.0   FotMob

In [21]:
def on7(proba):
    """A 5-league probability frame re-indexed onto matches7's rows."""
    keyed = proba.set_axis(pd.MultiIndex.from_frame(matches.loc[proba.index, KEY]))
    rows = top5_test7.set_index(KEY).index
    return pd.DataFrame(keyed.loc[rows].to_numpy(), index=top5_test7.index, columns=proba.columns)


probas7 = {
    "Pooled 7": walk_forward(matches7, make_model, FEATURES + INDICATORS7, TEST_SEASONS),
    "Per league": walk_forward(matches7, make_model, FEATURES, TEST_SEASONS, group_by="Div"),
    "Shipped 7": walk_forward_models(matches7, TEST_SEASONS),
    "Pooled 5": on7(probas["Pooled"]),
    "Shipped 5": on7(probas["Hybrid"]),
    "Bookmaker": bookmaker(test7),
}

base = []
for season in TEST_SEASONS:
    prior = matches7[matches7["Season"] < season]
    rates = prior.groupby("Div")["FTR"].value_counts(normalize=True).unstack()[LABELS]
    held = test7[test7["Season"] == season]
    base.append(pd.DataFrame(rates.loc[held["Div"]].to_numpy(), index=held.index, columns=LABELS))
probas7["Base rates"] = pd.concat(base).loc[test7.index]

# The shipped setup's E0 rows are the EPL-only model in both, and its other rows the pooled model
assert np.allclose(probas7["Shipped 7"].loc[top5_test7.index[top5_test7.Div == "E0"]],
                   probas7["Shipped 5"].loc[top5_test7.index[top5_test7.Div == "E0"]])

In [22]:
def compare7(frame, candidate, reference):
    """compare() over probas7."""
    global probas
    saved, probas = probas, probas7
    try:
        return compare(frame, candidate, reference)
    finally:
        probas = saved


models7 = ["Base rates", "Per league", "Pooled 7", "Bookmaker"]
rows = []
for div in DIVS7 + ["All"]:
    frame = test7 if div == "All" else test7[test7["Div"] == div]
    for name in models7:
        rows.append({"League": NAMES7.get(div, "All seven"), "Model": name, **scores(frame, probas7[name])})

by_league7 = pd.DataFrame(rows)
by_league7.pivot(index="League", columns="Model", values="logLoss")[models7].loc[[*NAMES7.values(), "All seven"]]

Model,Base rates,Per league,Pooled 7,Bookmaker
League,,,,
Premier League,1.0689,0.9799,0.9795,0.9605
La Liga,1.0639,0.9911,0.9847,0.9595
Bundesliga,1.0749,0.9991,0.9932,0.9697
Serie A,1.0869,0.9948,0.9928,0.9670
Ligue 1,1.0706,0.9989,0.9981,0.9784
Primeira Liga,1.0728,0.9414,0.9356,0.8984
Eredivisie,1.0720,0.9628,0.9548,0.9321
All seven,1.0729,0.9820,0.9780,0.9534


In [23]:
(
    by_league7[by_league7["Model"].isin(["Per league", "Pooled 7"])]
    .pivot(index="League", columns="Model", values=["accuracy", "roi"])
    .loc[[*NAMES7.values(), "All seven"]]
    .round(1)
)

accuracy                 roi         
Model          Per league Pooled 7 Per league Pooled 7
League                                                
Premier League       54.7     54.5        0.4      0.1
La Liga              52.3     53.8       -1.4      1.9
Bundesliga           51.6     51.4       -6.4     -7.0
Serie A              52.1     52.4       -3.5     -2.7
Ligue 1              51.6     51.7       -4.8     -4.4
Primeira Liga        55.2     55.6       -3.7     -3.1
Eredivisie           54.5     54.5       -6.5     -6.6
All seven            53.1     53.4       -3.5     -2.9

## Gate 1 - top 5: 7-league pooled vs 5-league pooled (non-inferior)

Both pooled models on all top-5 test matches. For the record, the same for
the shipped setup (E0 on its own model in both, so its E0 rows are
identical and only SP1/D1/I1/F1 can move).

In [24]:
gate7_1 = compare7(top5_test7, "Pooled 7", "Pooled 5")
print(f"Pooled 7 - pooled 5 on {gate7_1['matches']} top-5 matches: {gate7_1['mean']:+.4f} "
      f"[{gate7_1['ciLow']:+.4f}, {gate7_1['ciHigh']:+.4f}], P(better) {gate7_1['pBetter']:.2f}, "
      f"better in {gate7_1['seasonsBetter']}/{len(TEST_SEASONS)} seasons")
print(f"Gate 1 (non-inferior, upper < +{NON_INFERIOR_MARGIN}): {'PASS' if gate7_1['nonInferior'] else 'FAIL'}")

detail7 = [{"Comparison": "Shipped 7 - shipped 5", "League": "Top 5", **compare7(top5_test7, "Shipped 7", "Shipped 5")}]
for div in DIVS:
    frame = top5_test7[top5_test7["Div"] == div]
    detail7.append({"Comparison": "Pooled 7 - pooled 5", "League": NAMES7[div], **compare7(frame, "Pooled 7", "Pooled 5")})
pd.DataFrame(detail7).drop(columns=["better"])

Pooled 7 - pooled 5 on 7082 top-5 matches: -0.0001 [-0.0006, +0.0004], P(better) 0.65, better in 2/4 seasons
Gate 1 (non-inferior, upper < +0.002): PASS


,Comparison,League,matches,mean,ciLow,ciHigh,pBetter,seasonsBetter,nonInferior
0,Shipped 7 - shipped 5,Top 5,7082,-4.3987e-05,-0.0005,0.0004,0.5806,2,True
1,Pooled 7 - pooled 5,Premier League,1520,-2.6865e-04,-0.0015,0.0010,0.6752,2,True
2,Pooled 7 - pooled 5,La Liga,1520,7.9372e-05,-0.0011,0.0013,0.4478,2,True
3,Pooled 7 - pooled 5,Bundesliga,1224,-2.6425e-04,-0.0016,0.0012,0.6588,3,True
4,Pooled 7 - pooled 5,Serie A,1520,2.2304e-05,-0.0012,0.0012,0.4900,2,True
5,Pooled 7 - pooled 5,Ligue 1,1298,-1.0988e-04,-0.0013,0.0011,0.5848,2,True


## Gate 2 - P1 and N1: pooled vs their own model

Pooled must be better (CI upper < 0), or at least non-inferior
(upper < +0.002).

In [25]:
gate7_2 = {}
for div in NEW:
    frame = test7[test7["Div"] == div]
    gate7_2[div] = result = compare7(frame, "Pooled 7", "Per league")
    verdict = "better" if result["better"] else "non-inferior" if result["nonInferior"] else "FAIL"
    print(f"{NAMES7[div]}: pooled - own on {result['matches']} matches: {result['mean']:+.4f} "
          f"[{result['ciLow']:+.4f}, {result['ciHigh']:+.4f}], P(better) {result['pBetter']:.2f}, "
          f"better in {result['seasonsBetter']}/{len(TEST_SEASONS)} seasons -> {verdict}")

    y = frame["FTR"].to_numpy()
    diff = (per_match_log_loss(y, probas7["Pooled 7"].loc[frame.index][LABELS].to_numpy(), LABELS)
            - per_match_log_loss(y, probas7["Per league"].loc[frame.index][LABELS].to_numpy(), LABELS))
    sd = diff.std(ddof=1)
    print(f"    per-match SD {sd:.4f}; CI half-width ~{1.96 * sd / np.sqrt(len(diff)):.4f}; "
          f"an equally good model needs n ~ {(1.96 * sd / NON_INFERIOR_MARGIN) ** 2:,.0f} to pass non-inferiority")

both = test7[test7["Div"].isin(NEW)]
gate7_2["Both"] = compare7(both, "Pooled 7", "Per league")
print(f"Both together ({len(both)} matches, not a gate): {gate7_2['Both']['mean']:+.4f} "
      f"[{gate7_2['Both']['ciLow']:+.4f}, {gate7_2['Both']['ciHigh']:+.4f}]")

Primeira Liga: pooled - own on 1224 matches: -0.0058 [-0.0096, -0.0020], P(better) 1.00, better in 4/4 seasons -> better
    per-match SD 0.0675; CI half-width ~0.0038; an equally good model needs n ~ 4,380 to pass non-inferiority
Eredivisie: pooled - own on 1224 matches: -0.0080 [-0.0128, -0.0031], P(better) 1.00, better in 4/4 seasons -> better
    per-match SD 0.0870; CI half-width ~0.0049; an equally good model needs n ~ 7,271 to pass non-inferiority


Both together (2448 matches, not a gate): -0.0069 [-0.0100, -0.0039]


## Gate 3 - P1 and N1: pooled vs base rates (better), bookmaker alongside

In [26]:
gate7_3 = []
for div in NEW:
    frame = test7[test7["Div"] == div]
    for candidate, reference in [("Pooled 7", "Base rates"), ("Per league", "Base rates"), ("Pooled 7", "Bookmaker")]:
        gate7_3.append({"League": NAMES7[div], "Comparison": f"{candidate} - {reference}",
                        **compare7(frame, candidate, reference)})
gate7_3 = pd.DataFrame(gate7_3)
gate7_3.drop(columns=["nonInferior"])

,League,Comparison,matches,mean,ciLow,ciHigh,pBetter,seasonsBetter,better
0,Primeira Liga,Pooled 7 - Base rates,1224,-0.1373,-0.1611,-0.1130,1.0,4,True
1,Primeira Liga,Per league - Base rates,1224,-0.1314,-0.1570,-0.1047,1.0,4,True
2,Primeira Liga,Pooled 7 - Bookmaker,1224,0.0372,0.0264,0.0477,0.0,0,False
3,Eredivisie,Pooled 7 - Base rates,1224,-0.1172,-0.1408,-0.0928,1.0,4,True
4,Eredivisie,Per league - Base rates,1224,-0.1092,-0.1329,-0.0847,1.0,4,True
5,Eredivisie,Pooled 7 - Bookmaker,1224,0.0227,0.0106,0.0352,0.0,0,False


Season by season in P1 and N1, and the new leagues' indicator coefficients
(pooled model trained on 21-22 to 25-26, averaged over the calibration
folds, relative to the Premier League):

In [27]:
pd.DataFrame({
    (NAMES7[div], name): {
        season: log_loss(frame.loc[frame["Season"] == season, "FTR"],
                         probas7[name].loc[frame.index[frame["Season"] == season]][LABELS], labels=LABELS)
        for season in TEST_SEASONS
    }
    for div in NEW
    for frame in [test7[test7["Div"] == div]]
    for name in ["Per league", "Pooled 7", "Bookmaker"]
}).round(4)

Primeira Liga                    Eredivisie                   
         Per league Pooled 7 Bookmaker Per league Pooled 7 Bookmaker
22-23        0.9241   0.9079    0.8654     0.9555   0.9368    0.9195
23-24        0.9501   0.9453    0.9021     0.9174   0.9147    0.8939
24-25        0.9581   0.9560    0.9239     0.9781   0.9687    0.9376
25-26        0.9334   0.9331    0.9020     1.0002   0.9992    0.9776

In [28]:
train7 = matches7[matches7["Season"] < "26-27"]
fitted7 = make_model().fit(train7[FEATURES + INDICATORS7], train7["FTR"])
coef7 = np.mean([c.estimator.coef_ for c in fitted7.calibrated_classifiers_], axis=0)
pd.DataFrame(coef7, index=fitted7.classes_, columns=FEATURES + INDICATORS7).T.round(3)

,A,D,H
EloDiff,-0.003,0.000,0.003
XGDDiffLast5,-0.119,-0.051,0.170
GoalsPerGameDiff,-0.112,0.053,0.059
League_SP1,-0.127,0.075,0.052
League_D1,-0.028,0.028,0.000
League_I1,0.016,0.106,-0.122
League_F1,0.020,0.007,-0.027
League_P1,-0.027,0.055,-0.029
League_N1,-0.033,0.034,-0.001


## Promoted-team start for P1 and N1 (as P1-08)

The shipped setup over all seven leagues, Elo rebuilt per configuration:
`None`; pooled `P` (all seven leagues start promoted clubs at `1500 - P`);
and `P_L` for P1 or N1 alone. Chosen on 22-23/23-24, judged on 24-25/25-26
with P1-08's rule: `P_L` if it beats both `None` and the pooled value,
else the pooled value if it beats `None`, else `None`. The top 5 keep
P1-08's decision (`None`).

In [29]:
matches7["Promoted"], matches7["PromotedFirst10"] = promoted_flags(matches7)


def run7(promoted_elo):
    frame = matches7.copy()
    build_elo_feature(frame, promoted_elo=promoted_elo)
    return walk_forward_models(frame, ALL_TEST)


configs7 = {"None": None}
configs7.update({f"P={p} all": LEAGUE_BASE_ELO - p for p in P_GRID})
configs7.update({f"P={p} {div}": {div: LEAGUE_BASE_ELO - p} for div in NEW for p in P_GRID})

runs7 = {name: run7(value) for name, value in configs7.items()}

report7 = matches7[matches7["Season"].isin(REPORT)]
print(f"{len(runs7)} configurations")
report7[report7["Div"].isin(NEW)].groupby("Div")[["Promoted", "PromotedFirst10"]].sum().T

10 configurations


Div,N1,P1
Promoted,192,162
PromotedFirst10,57,46


In [30]:
tune7 = matches7[matches7["Season"].isin(TUNE)]
loss = lambda name, frame: log_loss(frame["FTR"], runs7[name].loc[frame.index][LABELS], labels=LABELS)

pooled_p7 = min(P_GRID, key=lambda p: loss(f"P={p} all", tune7))
league_p7 = {div: min(P_GRID, key=lambda p: loss(f"P={p} {div}", tune7[tune7.Div == div])) for div in NEW}
print(f"Pooled choice: P = {pooled_p7}; per league: {league_p7}")

pd.DataFrame({
    NAMES7[div]: {
        "None": loss("None", tune7[tune7.Div == div]),
        **{f"P={p}": loss(f"P={p} {div}", tune7[tune7.Div == div]) for p in P_GRID},
        **{f"P={p} all": loss(f"P={p} all", tune7[tune7.Div == div]) for p in P_GRID},
    }
    for div in NEW
}).T.round(4)

Pooled choice: P = 100; per league: {'P1': 100, 'N1': 150}


,None,P=100,P=150,P=200,P=100 all,P=150 all,P=200 all
Primeira Liga,0.9266,0.9267,0.9288,0.9316,0.9266,0.9289,0.9318
Eredivisie,0.9257,0.9232,0.9227,0.9229,0.9234,0.9232,0.9235


In [31]:
saved_runs, runs = runs, runs7  # diff_ci reads `runs`

rows, decisions7 = [], {}
pooled_name7 = f"P={pooled_p7} all"
for div in NEW + ["All"]:
    frame = report7 if div == "All" else report7[report7["Div"] == div]
    comparisons = [("Pooled P - None", pooled_name7, "None")]
    if div != "All":
        league_name = f"P={league_p7[div]} {div}"
        comparisons += [("P_L - None", league_name, "None"), ("P_L - pooled P", league_name, pooled_name7)]

    cis = {}
    for label, candidate, reference in comparisons:
        row = {"League": NAMES7.get(div, "All seven"), "Comparison": label,
               "P": pooled_p7 if candidate == pooled_name7 else league_p7.get(div), "n": len(frame)}
        for subset, mask in [("All matches", slice(None)), ("Promoted, season", frame["Promoted"]),
                             ("Promoted, first 10", frame["PromotedFirst10"])]:
            mean, ci = diff_ci(candidate, reference, frame[mask])
            row[subset] = fmt(mean, ci)
            if subset == "All matches":
                cis[label] = ci
        rows.append(row)

    if div != "All":
        if better(cis["P_L - None"]) and better(cis["P_L - pooled P"]):
            decisions7[div] = LEAGUE_BASE_ELO - league_p7[div]
        elif better(cis["Pooled P - None"]):
            decisions7[div] = LEAGUE_BASE_ELO - pooled_p7
        else:
            decisions7[div] = None

runs = saved_runs
print("promoted_elo decisions:", decisions7)
pd.DataFrame(rows)

promoted_elo decisions: {'P1': None, 'N1': None}


,League,Comparison,P,n,All matches,"Promoted, season","Promoted, first 10"
0,Primeira Liga,Pooled P - None,100,612,"-0.0030 [-0.0066, +0.0005]","-0.0100 [-0.0231, +0.0029]","-0.0259 [-0.0594, +0.0088]"
1,Primeira Liga,P_L - None,100,612,"-0.0032 [-0.0067, +0.0004]","-0.0099 [-0.0230, +0.0030]","-0.0259 [-0.0595, +0.0085]"
2,Primeira Liga,P_L - pooled P,100,612,"-0.0001 [-0.0005, +0.0002]","+0.0002 [-0.0006, +0.0009]","-0.0001 [-0.0014, +0.0012]"
3,Eredivisie,Pooled P - None,100,612,"-0.0030 [-0.0080, +0.0018]","-0.0073 [-0.0224, +0.0073]","-0.0028 [-0.0460, +0.0350]"
4,Eredivisie,P_L - None,150,612,"-0.0024 [-0.0075, +0.0027]","-0.0064 [-0.0220, +0.0090]","-0.0001 [-0.0468, +0.0426]"
5,Eredivisie,P_L - pooled P,150,612,"+0.0006 [-0.0011, +0.0024]","+0.0008 [-0.0044, +0.0062]","+0.0026 [-0.0105, +0.0165]"
6,All seven,Pooled P - None,100,4728,"-0.0015 [-0.0027, -0.0002]","-0.0044 [-0.0087, -0.0002]","-0.0125 [-0.0248, -0.0004]"


## Portugal and the Netherlands - conclusion

**Every gate passes: P1 and N1 join the pooled model, and `promoted_elo`
stays `None` for both.**

**Gate 1 - adding two leagues doesn't hurt the top 5.** 7-league pooled vs
5-league pooled on the 7,082 top-5 test matches is **-0.0001
[-0.0006, +0.0004]**, non-inferior with room to spare. Every league is
within +/-0.0003. For the shipped setup (E0 on its own model in both) it's
-0.00004 [-0.0005, +0.0004].

**Gate 2 - pooled beats each new league's own model, in all 4 seasons:**

| League | Own model | Pooled | Pooled - own [95% CI] | Bookmaker | Base rates |
|---|---|---|---|---|---|
| Primeira Liga | 0.9414 | 0.9356 | **-0.0058 [-0.0096, -0.0020]** | 0.8984 | 1.0728 |
| Eredivisie | 0.9628 | 0.9548 | **-0.0080 [-0.0128, -0.0031]** | 0.9321 | 1.0720 |

Both are *better*, not just non-inferior. That matters, because the
non-inferiority leg couldn't have been passed by an equally good model here.
The per-match SDs (0.068, 0.087) are higher than the EPL's 0.052, so an
equally good model would need ~4,400 / ~7,300 matches.

**Gate 3 - both beat their base rates by a wide margin** (P1 -0.137, N1
-0.117, CIs far from 0). The bookmaker is still well ahead: +0.037 in
Portugal and +0.023 in the Netherlands. These are the two most predictable
leagues for everyone (big-three dominance), so the absolute log losses are
lower than the top 5's.

**Promoted-team start - `None` for both.** Tuning chose P = 100 for P1 and
150 for N1 (pooled over seven: 100). On 24-25/25-26 neither league's gain
is clear of zero (P1 -0.0032 [-0.0067, +0.0004], N1 -0.0024
[-0.0075, +0.0027]). Each league has only ~160-190 promoted-club report
matches.

**Not a gate, for the record:** a pooled P = 100 for *all seven* leagues is
now just clear of zero on all 4,728 report matches (-0.0015
[-0.0027, -0.0002]), as are the promoted subsets. P1-08's version (five
leagues, P = 150) just missed (upper +0.0002). The plan's rule is per
league, so nothing changes. It's the strongest lead yet for the re-run after
26-27 finishes.

**Other findings:**
- **Flat-stake ROI** at average opening odds, own model -> pooled: P1 -3.7%
  -> -3.1%, N1 -6.5% -> -6.6%. Over all seven: -3.5% -> -2.9%.
- **Indicators:** P1 and N1 are small and close to the EPL (slightly more
  draws).
- **Data:** 100% FotMob xG in every P1/N1 league-season. The top 5's
  features are bit-identical with two more leagues in the frame (asserted).
